# Build a Condition-Aware Agent

**What you'll build:** A Claude agent that automatically identifies what conditions it needs before answering — and asks for them if they're missing.

**Time:** 15 minutes

**Learning objectives:**
1. See all five Claude API conditioning levers working together
2. Build the condition extraction → clarification → answer pipeline
3. Use forced `tool_choice` for structured outputs at each stage
4. Apply the condition stack to real question answering

**Prerequisites:**
- Module 2: Switch Variables
- Module 3: Condition Stack
- `ANTHROPIC_API_KEY` set in your environment

In [ ]:
learning_objectives(["A Claude agent that automatically identifies what conditions it needs before answering — and asks for them if they're missing", "15 minutes", "1. See all five Claude API conditioning levers working together", "Build the condition extraction → clarification → answer pipeline", "Use forced `tool_choice` for structured outputs at each stage", "Apply the condition stack to real question answering"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## Quick Win: The Agent in 90 Seconds

Here's the complete agent — 3 steps, each a Claude API call:

In [ ]:
import anthropic
import json
import os

client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY from environment

question = "Should I use a fixed-price contract or a time-and-materials contract for this project?"

# Step 1: What conditions do we need?
# Step 2: Ask for missing ones
# Step 3: Answer with full condition stack

print("Question:", question)
print("\nAgent: Let me figure out what I need to know to answer this well...")

## Step 1: Define the Condition Extraction Tool

The agent's first move: identify what switch variables the question requires.

This is Agent 1's job from Guide 02 — but here we'll do it as a single agent with multiple steps.

In [ ]:
# Schema for condition extraction output
# The agent identifies: which conditions it needs, which are present, which are missing

CONDITION_EXTRACTION_SCHEMA = {
    "type": "object",
    "required": ["conditions_needed", "conditions_present", "conditions_missing", "can_answer_now"],
    "properties": {
        "conditions_needed": {
            "type": "array",
            "description": "All switch variables this question requires for a precise answer",
            "items": {
                "type": "object",
                "required": ["name", "why_it_matters", "question_to_ask"],
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Short name for this condition (e.g., 'project_scope_certainty')"
                    },
                    "why_it_matters": {
                        "type": "string",
                        "description": "How this condition changes the answer"
                    },
                    "question_to_ask": {
                        "type": "string",
                        "description": "Natural language question to ask the user"
                    }
                }
            }
        },
        "conditions_present": {
            "type": "object",
            "description": "Conditions already present in the question as key-value pairs"
        },
        "conditions_missing": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Names of conditions not yet specified by the user"
        },
        "can_answer_now": {
            "type": "boolean",
            "description": "True only if all critical switch variables are already present"
        }
    }
}

# System prompt for the extraction step
# Notice: this is Layer 0 — persistent role and objective for this step
EXTRACTION_SYSTEM = """You are a condition extraction specialist.

Your role: analyze questions and identify which switch variables are needed for a precise answer.

A switch variable is a condition that, if changed, would change which answer is correct.
Example: For 'should I take this job offer?' — salary, career trajectory, and relocation
are switch variables. The asker's name is not.

Be specific: identify the 2-4 most important switch variables, not an exhaustive list.
If a condition is already stated in the question, mark it as present."""

print("Schemas and system prompts defined.")

In [ ]:
def extract_conditions(question: str) -> dict:
    """
    Step 1: Extract what switch variables the question needs.
    
    Uses forced tool_choice to guarantee structured JSON output.
    Returns the extraction result as a dict.
    """
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1500,
        system=EXTRACTION_SYSTEM,  # Layer 0: persistent role
        tools=[{
            "name": "submit_extraction",
            "description": "Submit your analysis of what conditions this question needs.",
            "input_schema": CONDITION_EXTRACTION_SCHEMA
        }],
        # Force structured output — no natural language response allowed
        tool_choice={"type": "tool", "name": "submit_extraction"},
        messages=[{
            "role": "user",
            "content": f"Analyze this question and identify what conditions are needed:\n\n{question}"
        }]
    )
    
    # Extract the structured result from tool_use block
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_extraction":
            return block.input
    
    raise ValueError("Extraction step did not return structured output")


# Run the extraction
extraction = extract_conditions(question)

print("Conditions needed:")
for cond in extraction["conditions_needed"]:
    present = cond["name"] not in extraction["conditions_missing"]
    status = "[present]" if present else "[MISSING]"
    print(f"  {status} {cond['name']}: {cond['why_it_matters']}")

print(f"\nCan answer now: {extraction['can_answer_now']}")
print(f"Conditions already present: {extraction['conditions_present']}")

## Step 2: Ask for Missing Conditions

The agent now asks focused questions — one per missing switch variable.

This step assembles the complete condition stack before generating the final answer.

In [ ]:
def gather_missing_conditions(extraction: dict, simulated_answers: dict = None) -> dict:
    """
    Step 2: Ask the user for each missing condition.
    
    In production, this would be an interactive loop.
    Here, we accept simulated_answers to demonstrate the flow without a live user.
    
    Returns the assembled condition stack.
    """
    # Start with conditions already present in the question
    condition_stack = dict(extraction["conditions_present"])
    
    # For each missing condition, either ask or use simulated answer
    for condition_name in extraction["conditions_missing"]:
        # Find the question to ask for this condition
        question_to_ask = None
        for cond in extraction["conditions_needed"]:
            if cond["name"] == condition_name:
                question_to_ask = cond["question_to_ask"]
                break
        
        if question_to_ask:
            print(f"Agent: {question_to_ask}")
            
            if simulated_answers and condition_name in simulated_answers:
                # Use simulated answer in notebook demo
                answer = simulated_answers[condition_name]
                print(f"User: {answer}")
                condition_stack[condition_name] = answer
            else:
                # Interactive mode — ask the actual user
                answer = input("Your answer: ")
                condition_stack[condition_name] = answer
            print()
    
    return condition_stack


# Simulated user answers for demonstration
# In real use, remove simulated_answers and the agent asks the live user
simulated_user_answers = {
    "project_scope_certainty": "The scope is moderately uncertain — we know the main deliverables but expect some changes",
    "client_risk_tolerance": "The client prefers cost certainty over flexibility — they have a fixed budget",
    "timeline_flexibility": "Timeline is firm — must deliver by Q4 regardless",
    "contractor_relationship": "New contractor, first engagement with this vendor"
}

print("=== Condition Gathering Phase ===")
print()
condition_stack = gather_missing_conditions(extraction, simulated_user_answers)

print("=== Assembled Condition Stack ===")
for k, v in condition_stack.items():
    print(f"  {k}: {v}")

## Step 3: Generate the Final Answer

Now the agent has a full condition stack. It generates an answer conditioned on the specific conditions — not the average case.

In [ ]:
# Schema for the final answer
# Structured output ensures the answer is explicit about which conditions drove it
ANSWER_SCHEMA = {
    "type": "object",
    "required": ["recommendation", "reasoning", "conditions_that_drove_this_answer", "confidence"],
    "properties": {
        "recommendation": {
            "type": "string",
            "description": "The direct recommendation"
        },
        "reasoning": {
            "type": "string",
            "description": "Explanation of why, given the conditions"
        },
        "conditions_that_drove_this_answer": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "condition": {"type": "string"},
                    "value":     {"type": "string"},
                    "effect":    {"type": "string", "description": "How this condition changed the recommendation"}
                }
            }
        },
        "if_conditions_were_different": {
            "type": "string",
            "description": "What the recommendation would be if the main switch variable flipped"
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"]
        }
    }
}

ANSWERER_SYSTEM = """You are a precise reasoning agent.

You receive questions together with a set of specified conditions.
Your job: generate an answer that is precisely conditioned on those values.

Never answer as if you are advising the average case.
Always answer as if the specified conditions are known facts that constrain the solution space.
Explicitly show which conditions drove your recommendation."""


def generate_conditioned_answer(question: str, condition_stack: dict) -> dict:
    """
    Step 3: Generate the answer conditioned on the full condition stack.
    
    Injects conditions via the system prompt (Layer 0) and user message (Layer 5).
    Uses forced tool_choice for structured output.
    """
    # Format condition stack for injection into user message
    conditions_text = json.dumps(condition_stack, indent=2)
    
    # User message includes both the question and the conditions
    # Conditions are Layer 5 (task-specific) — they go in the user message
    user_message = f"""Question: {question}

Conditions specified for this question:
{conditions_text}

Answer the question with full awareness of these conditions.
Show explicitly how each condition influenced your recommendation."""
    
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=2000,
        system=ANSWERER_SYSTEM,  # Layer 0: stable role
        tools=[{
            "name": "submit_answer",
            "description": "Submit your structured, conditioned answer.",
            "input_schema": ANSWER_SCHEMA
        }],
        tool_choice={"type": "tool", "name": "submit_answer"},  # Force structured output
        messages=[{"role": "user", "content": user_message}]
    )
    
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_answer":
            return block.input
    
    raise ValueError("Answer step did not return structured output")


answer = generate_conditioned_answer(question, condition_stack)

print("=== Final Answer ===")
print(f"Recommendation: {answer['recommendation']}")
print(f"\nReasoning: {answer['reasoning']}")
print(f"\nConfidence: {answer['confidence']}")
print(f"\nIf conditions were different: {answer['if_conditions_were_different']}")
print("\nConditions that drove this answer:")
for cond in answer["conditions_that_drove_this_answer"]:
    print(f"  - {cond['condition']} = '{cond['value']}' → {cond['effect']}")

## The Complete Agent: All Three Steps Together

Here is the full pipeline as a single reusable function:

In [ ]:
def condition_aware_agent(
    question: str,
    known_conditions: dict = None,
    interactive: bool = False
) -> dict:
    """
    A complete condition-aware agent.
    
    Pipeline:
    1. Extract what conditions the question needs
    2. If conditions are missing and interactive=True, ask the user
       If not interactive, proceed with what's available
    3. Generate answer conditioned on all available conditions
    
    Parameters:
        question:         The user's question
        known_conditions: Pre-specified conditions (e.g., from a form or session context)
        interactive:      If True, prompt the user for missing conditions
    
    Returns:
        dict with keys: extraction, condition_stack, answer
    """
    known_conditions = known_conditions or {}
    
    # Step 1: Extract needed conditions
    print("[Step 1] Analyzing what conditions this question needs...")
    extraction = extract_conditions(question)
    
    missing = extraction["conditions_missing"]
    print(f"  Found {len(extraction['conditions_needed'])} needed conditions")
    print(f"  Missing: {missing}")
    
    # Step 2: Assemble condition stack
    print("\n[Step 2] Assembling condition stack...")
    condition_stack = dict(extraction["conditions_present"])
    condition_stack.update(known_conditions)  # Add pre-specified conditions
    
    if interactive and missing:
        # Ask user for remaining missing conditions
        condition_stack = gather_missing_conditions(extraction, simulated_answers=None)
    elif missing:
        print(f"  Note: {len(missing)} conditions unavailable — proceeding with partial stack")
        print(f"  Answer confidence may be lower than if all conditions were specified")
    
    # Step 3: Generate conditioned answer
    print("\n[Step 3] Generating answer conditioned on available conditions...")
    answer = generate_conditioned_answer(question, condition_stack)
    
    print(f"  Confidence: {answer['confidence']}")
    
    return {
        "extraction":       extraction,
        "condition_stack":  condition_stack,
        "answer":           answer
    }


# Test with a different question and pre-specified conditions
result = condition_aware_agent(
    question="What database technology should I use for this application?",
    known_conditions={
        "scale": "10 million users, 50 million rows",
        "team_expertise": "Strong PostgreSQL background, limited NoSQL experience",
        "budget": "Startup, cost-sensitive — managed cloud preferred"
    },
    interactive=False  # Set to True to ask live questions
)

print("\n=== Result ===")
print(f"Recommendation: {result['answer']['recommendation']}")
print(f"Reasoning: {result['answer']['reasoning']}")

## Comparison: With Conditions vs. Without

The most important thing to observe: how different is the answer when conditions are missing?

In [ ]:
# Ask the same question with and without conditions
# This demonstrates the posterior shift that conditions produce

test_question = "Should I raise a Series A round now or wait?"

# Version 1: No conditions
print("=== Version 1: No Conditions ===")
no_conditions_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=400,
    messages=[{"role": "user", "content": test_question}]
)
print(no_conditions_response.content[0].text[:500])

print()
print("=" * 60)
print()

# Version 2: Full condition stack
print("=== Version 2: Full Condition Stack ===")
conditions = {
    "current_arr": "$1.2M ARR, growing 15% MoM for 6 months",
    "runway": "14 months at current burn",
    "market_timing": "Q1 2025, public SaaS multiples at 8x ARR",
    "strategic_objective": "Build distribution partnership before raising — need 3 more months",
    "competitive_pressure": "One well-funded competitor just closed Series B"
}

conditions_text = json.dumps(conditions, indent=2)
conditioned_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=400,
    messages=[{
        "role": "user",
        "content": f"""Question: {test_question}

Conditions:
{conditions_text}

Answer given these specific conditions."""
    }]
)
print(conditioned_response.content[0].text[:500])

print()
print("Observe: Version 1 gives general advice about Series A timing.")
print("Version 2 reasons about this specific company's specific situation.")

## Modify This: Try Your Own Questions

Run the condition-aware agent on a question from your own domain:

In [ ]:
# EXPERIMENT 1: A question from your domain
# Replace with any question that has a context-dependent answer
your_question = "Should I refactor this codebase now or ship the feature first?"

# Run without pre-specified conditions — see what the agent asks for
# (it will skip the interactive clarification questions in non-interactive mode,
# but you'll see what it would ask)
extraction_only = extract_conditions(your_question)

print("For your question, the agent would ask:")
for cond in extraction_only["conditions_needed"]:
    print(f"  - {cond['question_to_ask']}")

print(f"\nConditions it found already present: {extraction_only['conditions_present']}")

In [ ]:
# EXPERIMENT 2: Pre-specify conditions and see confidence change
# Try providing partial conditions vs. full conditions
# Observe how confidence changes and how the answer changes

# Partial conditions
partial = condition_aware_agent(
    question=your_question,
    known_conditions={"team_size": "4 engineers"}  # Only one condition specified
)
print(f"Partial conditions confidence: {partial['answer']['confidence']}")

# YOUR CODE: Add more conditions and observe confidence and answer changes
# full_conditions = {
#     "team_size": "4 engineers",
#     "...": "..."
# }
# full = condition_aware_agent(question=your_question, known_conditions=full_conditions)

## Summary

**What you built:** A three-step condition-aware agent that:
1. Identifies which switch variables a question needs (Step 1)
2. Gathers missing conditions interactively or from pre-specified context (Step 2)
3. Generates an answer explicitly conditioned on the assembled stack (Step 3)

**Claude API patterns used:**
- `system` parameter for persistent role (Layer 0)
- `tool_choice: forced` for guaranteed structured output at every step
- Structured JSON schemas that travel alongside outputs
- Explicit condition injection in user messages (Layer 5)

**Key observation:**
The version with conditions produces an answer calibrated to a specific situation. The version without conditions produces an answer calibrated to the average situation. In any high-stakes domain, you are not the average case.

**Next:** `02_multi_agent_pipeline.ipynb` — split this into two specialized agents that pass switch variables via JSON handoff.